### Quantium Virtual Internship - Stores Performance Analysis - Task 2

In [1]:
#Loading needed libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
import re
import matplotlib.pyplot

In [2]:
print(os.getcwd())

/home/leda/Projects/Performance_stores


In [5]:
print(os.listdir())

['Untitled.ipynb', 'Task 2.docx', '.git', 'QVI_data.csv', '.ipynb_checkpoints', 'QVI_data.csv:Zone.Identifier']


In [7]:
#Importing the dataset
df = pd.read_csv("QVI_data.csv")

In [8]:
df.head()

,LYLTY_CARD_NBR,DATE,STORE_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER
0,1000,2018-10-17,1,1,5,Natural Chip Compny SeaSalt175g,2,6.0,175,NATURAL,YOUNG SINGLES/COUPLES,Premium
1,1002,2018-09-16,1,2,58,Red Rock Deli Chikn&Garlic Aioli 150g,1,2.7,150,RRD,YOUNG SINGLES/COUPLES,Mainstream
2,1003,2019-03-07,1,3,52,Grain Waves Sour Cream&Chives 210G,1,3.6,210,GRNWVES,YOUNG FAMILIES,Budget
3,1003,2019-03-08,1,4,106,Natural ChipCo Hony Soy Chckn175g,1,3.0,175,NATURAL,YOUNG FAMILIES,Budget
4,1004,2018-11-02,1,5,96,WW Original Stacked Chips 160g,1,1.9,160,WOOLWORTHS,OLDER SINGLES/COUPLES,Mainstream


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264834 entries, 0 to 264833
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   LYLTY_CARD_NBR    264834 non-null  int64  
 1   DATE              264834 non-null  object 
 2   STORE_NBR         264834 non-null  int64  
 3   TXN_ID            264834 non-null  int64  
 4   PROD_NBR          264834 non-null  int64  
 5   PROD_NAME         264834 non-null  object 
 6   PROD_QTY          264834 non-null  int64  
 7   TOT_SALES         264834 non-null  float64
 8   PACK_SIZE         264834 non-null  int64  
 9   BRAND             264834 non-null  object 
 10  LIFESTAGE         264834 non-null  object 
 11  PREMIUM_CUSTOMER  264834 non-null  object 
dtypes: float64(1), int64(6), object(5)
memory usage: 24.2+ MB


In [17]:
# Convert everything to numeric where possible (others become NaN)
numeric_dates = pd.to_numeric(df['DATE'], errors='coerce')

# Step 1: handle Excel serial dates (numeric only)
df['DATE_converted'] = pd.to_datetime(numeric_dates, origin='1899-12-30', unit='D',errors='coerce')

# Step 2: fill remaining with normal parsing (strings)
mask = df['DATE_converted'].isna()

df.loc[mask, 'DATE_converted'] = pd.to_datetime(
    df.loc[mask, 'DATE'],
    errors='coerce'
)

# Replace original column
df['DATE'] = df['DATE_converted']
df.drop(columns=['DATE_converted'], inplace=True)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264834 entries, 0 to 264833
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   LYLTY_CARD_NBR    264834 non-null  int64         
 1   DATE              264834 non-null  datetime64[ns]
 2   STORE_NBR         264834 non-null  int64         
 3   TXN_ID            264834 non-null  int64         
 4   PROD_NBR          264834 non-null  int64         
 5   PROD_NAME         264834 non-null  object        
 6   PROD_QTY          264834 non-null  int64         
 7   TOT_SALES         264834 non-null  float64       
 8   PACK_SIZE         264834 non-null  int64         
 9   BRAND             264834 non-null  object        
 10  LIFESTAGE         264834 non-null  object        
 11  PREMIUM_CUSTOMER  264834 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(6), object(4)
memory usage: 24.2+ MB


In [19]:
df['YEARMONTH'] = df['DATE'].dt.year * 100 + df['DATE'].dt.month

In [22]:
df.head(10)

,LYLTY_CARD_NBR,DATE,STORE_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER,YEARMONTH
0,1000,2018-10-17,1,1,5,Natural Chip Compny SeaSalt175g,2,6.0,175,NATURAL,YOUNG SINGLES/COUPLES,Premium,201810
1,1002,2018-09-16,1,2,58,Red Rock Deli Chikn&Garlic Aioli 150g,1,2.7,150,RRD,YOUNG SINGLES/COUPLES,Mainstream,201809
2,1003,2019-03-07,1,3,52,Grain Waves Sour Cream&Chives 210G,1,3.6,210,GRNWVES,YOUNG FAMILIES,Budget,201903
3,1003,2019-03-08,1,4,106,Natural ChipCo Hony Soy Chckn175g,1,3.0,175,NATURAL,YOUNG FAMILIES,Budget,201903
4,1004,2018-11-02,1,5,96,WW Original Stacked Chips 160g,1,1.9,160,WOOLWORTHS,OLDER SINGLES/COUPLES,Mainstream,201811
5,1005,2018-12-28,1,6,86,Cheetos Puffs 165g,1,2.8,165,CHEETOS,MIDAGE SINGLES/COUPLES,Mainstream,201812
6,1007,2018-12-04,1,7,49,Infuzions SourCream&Herbs Veg Strws 110g,1,3.8,110,INFUZIONS,YOUNG SINGLES/COUPLES,Budget,201812
7,1007,2018-12-05,1,8,10,RRD SR Slow Rst Pork Belly 150g,1,2.7,150,RRD,YOUNG SINGLES/COUPLES,Budget,201812
8,1009,2018-11-20,1,9,20,Doritos Cheese Supreme 330g,1,5.7,330,DORITOS,NEW FAMILIES,Premium,201811
9,1010,2018-09-09,1,10,51,Doritos Mexicana 170g,2,8.8,170,DORITOS,YOUNG SINGLES/COUPLES,Mainstream,201809


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264834 entries, 0 to 264833
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   LYLTY_CARD_NBR    264834 non-null  int64         
 1   DATE              264834 non-null  datetime64[ns]
 2   STORE_NBR         264834 non-null  int64         
 3   TXN_ID            264834 non-null  int64         
 4   PROD_NBR          264834 non-null  int64         
 5   PROD_NAME         264834 non-null  object        
 6   PROD_QTY          264834 non-null  int64         
 7   TOT_SALES         264834 non-null  float64       
 8   PACK_SIZE         264834 non-null  int64         
 9   BRAND             264834 non-null  object        
 10  LIFESTAGE         264834 non-null  object        
 11  PREMIUM_CUSTOMER  264834 non-null  object        
 12  YEARMONTH         264834 non-null  int32         
dtypes: datetime64[ns](1), float64(1), int32(1), int64(6), objec

In [34]:
#Define the measure calculations to use during the analysis
df_measure_overtime = (df.groupby(["STORE_NBR","YEARMONTH"])
    .agg(
        total_sales = ("TOT_SALES","sum"),
        num_customers = ("LYLTY_CARD_NBR", "nunique"),
        num_txn = ("TXN_ID", "nunique"),
        num_prod = ("PROD_QTY", "sum")
        )
    .reset_index()
)                        
    

In [35]:
df_measure_overtime.head(10)

,STORE_NBR,YEARMONTH,total_sales,num_customers,num_txn,num_prod
0,1,201807,206.9,49,52,62
1,1,201808,176.1,42,43,54
2,1,201809,278.8,59,62,75
3,1,201810,188.1,44,45,58
4,1,201811,192.6,46,47,57
5,1,201812,189.6,42,47,57
6,1,201901,154.8,35,36,42
7,1,201902,225.4,52,55,65
8,1,201903,192.9,45,49,58
9,1,201904,192.9,42,43,57


In [36]:
#Calculating transactions per costumer
df_measure_overtime["num_txn_per_costumer"] = ( df_measure_overtime["num_txn"] /
                                                df_measure_overtime["num_customers"])

#Calculating chips per costumer
df_measure_overtime["num_chips_per_costumer"] = ( df_measure_overtime["num_prod"] /
                                                df_measure_overtime["num_customers"])

#Calculating average price per unit
df_measure_overtime["avg_price_per_unit"] = ( df_measure_overtime["total_sales"] /
                                                df_measure_overtime["num_prod"])

In [39]:
df_measure_overtime = df_measure_overtime.sort_values(['STORE_NBR', 'YEARMONTH'])

In [42]:
#Filtering the data to the pre-trail period and stores with full observation periods
store_counts = (
    df_measure_overtime
    .groupby("STORE_NBR")
    .size()
    .reset_index(name = "Number")
)

store_counts.head()

,STORE_NBR,Number
0,1,12
1,2,12
2,3,12
3,4,12
4,5,12


In [46]:
#Keeping the stores with 12 observations
stores_full_obs = store_counts.loc[store_counts["Number"] == 12, "STORE_NBR"]

#Filtering for the pre-trial period, trial period (February 2019 to the end of April 2019)
pre_trial_measures = df_measure_overtime[
    (df_measure_overtime['YEARMONTH'] < 201902) &
    (df_measure_overtime['STORE_NBR'].isin(stores_full_obs))
]

In [ ]:
#Finding correlation of a trial store with the control stores
def calculate_correlation(df, trial_store, measure):
    correlations = []

    # Get trial store data
    trial_df = df_measure_overtime[df_measure_overtime['STORE_NBR'] == trial_store]
        [['YEARMONTH', measure]]

    for store in df_measure_overtime['STORE_NBR'].unique():
        if store == trial_store:
            continue  # skip itself

        control_df = df[df['STORE_NBR'] == store][['YEARMONTH', measure]]

        # Merge on YEARMONTH to align time periods
        merged = pd.merge(trial_df, control_df, on='YEARMONTH', suffixes=('_trial', '_control'))

        # Calculate correlation
        if len(merged) > 1:
            corr = merged[f'{measure}_trial'].corr(merged[f'{measure}_control'])
        else:
            corr = None

        correlations.append({
            'STORE_NBR': store,
            'correlation': corr
        })

    return pd.DataFrame(correlations)